# Кредитный скоринг физических лиц
Учебный проект LendingClub. **Ограничение:** `Charged Off` — прокси списания, не доказательство просрочки 90+ дней. Статусы завершённых кредитов могут приводить к смещению выборки. Все расчёты прибыли условные. Не используйте для реальных кредитных решений.

Ноутбуки запускаются последовательно 01 → 02 → 03 → 04. Каталог `artifacts/` сохраняет промежуточные результаты. Один раз разместите архив `accepted_2007_to_2018Q4.csv.gz` в текущем каталоге Colab или укажите путь в ячейке ниже. При новом сеансе Colab локальные файлы пропадают: используйте Google Drive или загрузите готовые промежуточные файлы.

Зависимости: `pip install pandas numpy matplotlib seaborn scikit-learn lightgbm optbinning shap joblib pyyaml`.

## 03. LightGBM и проверка калибровки на validation

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
os.makedirs('artifacts', exist_ok=True)
from sklearn.metrics import roc_auc_score, roc_curve
def metrics(y,p):
    auc=roc_auc_score(y,p);fpr,tpr,_=roc_curve(y,p)
    return {'ROC-AUC':auc,'Gini':2*auc-1,'KS':float(np.max(tpr-fpr))}
def profit(y,p,loan,threshold,good_return=.15,lgd=.45):
    y=np.asarray(y); p=np.asarray(p); loan=np.asarray(loan)
    return float(np.where(y==0,good_return*loan,-lgd*loan)[p<threshold].sum())

import lightgbm as lgb
from sklearn.calibration import calibration_curve
b=joblib.load('artifacts/baseline.pkl')
train=pd.read_pickle('artifacts/train.pkl');valid=pd.read_pickle('artifacts/valid.pkl')
def create_features(data):
    d=data.copy()
    num=['annual_inc','loan_amnt','installment','dti','fico_range_low','fico_range_high','revol_bal','open_acc','total_acc','delinq_2yrs']
    for c in num: d[c]=pd.to_numeric(d[c], errors='coerce')
    d['income_monthly']=d['annual_inc']/12
    d['loan_to_income']=d['loan_amnt']/(d['annual_inc']+1)
    d['payment_to_income']=d['installment']/(d['income_monthly']+1)
    d['debt_to_income']=d['dti']/100
    d['fico_average']=(d['fico_range_low']+d['fico_range_high'])/2
    d['fico_spread']=d['fico_range_high']-d['fico_range_low']
    d['credit_per_account']=d['revol_bal']/(d['open_acc']+1)
    d['account_activity']=d['open_acc']/(d['total_acc']+1)
    d['delinq_per_account']=d['delinq_2yrs']/(d['total_acc']+1)
    d['revolving_utilization']=pd.to_numeric(d['revol_util'].astype(str).str.replace('%','',regex=False),errors='coerce')/100
    return d

train=create_features(train);valid=create_features(valid)
f=b['features'];binning=b['binning']
Xt=binning.transform(train[f].replace([np.inf,-np.inf],np.nan),metric='woe')
Xv=binning.transform(valid[f].replace([np.inf,-np.inf],np.nan),metric='woe')
y_train=train.target;y_valid=valid.target
model=lgb.LGBMClassifier(n_estimators=1000,learning_rate=.03,num_leaves=31,random_state=42,verbosity=-1)
model.fit(Xt,y_train,eval_set=[(Xv,y_valid)],eval_metric='auc',callbacks=[lgb.early_stopping(50,verbose=False)])
p_boost=model.predict_proba(Xv)[:,1]
p_base=b['base_model'].predict_proba(Xv)[:,1]
display(pd.DataFrame([metrics(y_valid,p_base),metrics(y_valid,p_boost)],index=['Logistic regression','LightGBM']))


In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
for label,p in [('Logistic',p_base),('LightGBM',p_boost)]:
    actual,pred=calibration_curve(y_valid,p,n_bins=10,strategy='quantile')
    ax.plot(pred,actual,'o-',label=label)
ax.plot([0,1],[0,1],'--',color='gray');ax.legend();ax.set(xlabel='Предсказанная вероятность',ylabel='Фактическая доля',title='Калибровочная диагностика');plt.show()
# Диагностика != калибровка. Для настройки вероятностей требуется отдельное подмножество validation.
joblib.dump({'model':model,'binning':binning,'features':f},'artifacts/model_bundle.pkl')
np.save('artifacts/p_boost_valid.npy',p_boost)
print('Промежуточная модель сохранена')


**Ограничение:** вероятность не откалибрована. Требование задания о фактической калибровке `CalibratedClassifierCV` / isotonic ещё остаётся открытым. Не считайте `PD × LGD × EAD` валидной денежной оценкой без этого шага.